---

## 🎭 PASO 1.5: Selección de Nivel de Personalidad (NUEVO v2.3.0)

El sistema ahora soporta **dos niveles** de personalidad:

### 📋 NIVEL BASE (`level="base"`)
- **Voz técnica neutral**, sin proyectos personales
- Stack-agnostic con múltiples opciones tecnológicas
- Basado en evidencia pública (benchmarks, papers)
- **Ideal para:** Actualidad, comparativas, explicaciones técnicas

### 🎯 NIVEL FULL (`level="full"`) - Default
- **Voz con experiencia personal** y stack específico
- Referencias a proyectos (Cofares, consultoría)
- Primera persona en retrospectivas
- **Ideal para:** Tutoriales, case studies, lecciones aprendidas

**Guía completa:** Ver `GHEN/README_PERSONALIDAD.md`

---

# 🚀 Pipeline GHEN Digital - Generador de Contenidos Técnicos

Este notebook implementa 3 métodos para generar contenido técnico de calidad:

1. **Por Tópico**: Define un tema técnico amplio y el sistema sugerirá una keyword óptima
2. **Por Keyword**: Usa directamente una keyword específica
3. **Por Newsletter/URL**: Analiza contenido de newsletters tech para generar artículos

---

## 📦 Configuración Inicial

In [1]:
# Importar librerías
import sys
import os
from dotenv import load_dotenv
import pandas as pd
import importlib

# Cargar variables de entorno
load_dotenv()

# Crear directorio outputs si no existe
os.makedirs('outputs', exist_ok=True)

# Importar módulos personalizados
from longcontent_generator import core, scraper, utils, config

# 🔄 Recargar módulos si ya fueron importados (útil después de cambios)
# Forzar recarga de gmail si existe en memoria
if 'longcontent_generator.gmail' in sys.modules:
    import longcontent_generator.gmail
    importlib.reload(longcontent_generator.gmail)
    print("🔄 Módulo gmail recargado")

importlib.reload(core)

print("✅ Librerías importadas correctamente")
print(f"📊 Modelo Gemini: {config.CONFIG['gemini_model']}")
print(f"📁 Outputs en: {config.CONFIG['output_dir']}")
print("🔄 Módulo core recargado")

/Users/gabrielnoguera/Documents/ghen/LongContent_Generator_Script/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Configuración Gemini cargada correctamente
📊 Configuración cargada:
   • Modelo: gemini-2.5-flash
   • Temperatura: 0.8
   • País por defecto: ES
✅ LongContent Generator v2.3.0 cargado correctamente
📋 Funciones principales disponibles:
   • google_custom_search() - Búsqueda en Google
   • scrape_articles_batch() - Scraping de artículos
   • analyze_articles_batch() - Análisis SEO con Gemini
   • generate_article_from_outline() - Generación de contenido
   • qa_article_coverage() - Análisis de calidad
   • publish_article_from_markdown_cleaned() - Publicación WordPress
   • extract_newsletter_from_gmail() - 🆕 Leer newsletters desde Gmail
   • list_gmail_newsletters() - 🆕 Listar newsletters disponibles
   • add_source_links_to_article() - 🆕 Referencias automáticas
   • generate_featured_image_prompt() - 🆕 Generar prompt de imagen
   • generate_image_from_prompt() - 🆕 Generar imagen con Imagen 3
   • load_ghen_context(level='base|full') - 🎭 Sistema híbrido de personalidad
🔄 Módulo gmail

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🔧 SELECCIONA EL NIVEL DE PERSONALIDAD
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# "base" → Voz técnica neutral, sin proyectos personales (actualidad, comparativas)
# "full" → Voz con experiencia personal y stack específico (tutoriales, case studies)
# Ver GHEN/README_PERSONALIDAD.md para más detalles

nivel_personalidad = "base"  # Cambiar a "base" o "full"

# Cargar contexto de GHEN (personalidad técnica)
ghen_context = core.load_ghen_context(level=nivel_personalidad)

print("\n📋 Contexto de GHEN cargado:")
print(f"   • Nivel: {ghen_context['level'].upper()}")
print(f"   • Personalidad: {len(ghen_context['personality'])} caracteres")
if ghen_context['project']:
    print(f"   • Proyecto: {len(ghen_context['project'])} caracteres")
if ghen_context['audience']:
    print(f"   • Audiencia: {len(ghen_context['audience'])} caracteres")

print("\n💡 GUÍA DE NIVELES:")
if ghen_context['level'] == "base":
    print("   BASE: Contenido neutral, objetivo, stack-agnostic")
    print("   Ideal para: actualidad, comparativas, explicaciones técnicas")
else:
    print("   FULL: Contenido con experiencia personal, voice marca GHEN")
    print("   Ideal para: tutoriales, case studies, lecciones aprendidas")

✅ Personalidad técnica de GHEN cargada (NIVEL BASE: neutral, sin proyectos personales)

📋 Contexto de GHEN cargado:
   • Nivel: BASE
   • Personalidad: 5419 caracteres

💡 GUÍA DE NIVELES:
   BASE: Contenido neutral, objetivo, stack-agnostic
   Ideal para: actualidad, comparativas, explicaciones técnicas


---

## 🎯 SELECCIONA TU MÉTODO DE CREACIÓN

Cambia el valor de `metodo_seleccionado` a una de estas opciones:
- `"topico"` - Generar a partir de un tema amplio
- `"keyword"` - Generar a partir de una keyword específica
- `"newsletter"` - Generar a partir de una URL de newsletter

**Ejecuta solo UNA de las secciones según tu elección.**

In [3]:
# 🔧 CONFIGURA AQUÍ TU MÉTODO
metodo_seleccionado = "newsletter"  # Cambia a "topico", "keyword" o "newsletter"

print(f"🎯 Método seleccionado: {metodo_seleccionado.upper()}")

🎯 Método seleccionado: NEWSLETTER


---

# 📝 MÉTODO 1: Generación por Tópico

Define un tema técnico amplio y el sistema sugerirá la mejor keyword para SEO.

In [4]:
if metodo_seleccionado == "topico":
    # Define tu tópico técnico aquí
    topico = "Implementación de agentes ReAct con LangGraph para sistemas de producción"
    
    print(f"📌 Tópico definido: {topico}")
    print("\n🤖 Analizando el tópico y sugiriendo una keyword óptima...\n")
    
    # Sugerir keyword desde el tópico
    keyword_principal = core.suggest_keyword_from_topic(topico, ghen_context)
    
    if keyword_principal:
        print(f"\n✨ Keyword sugerida: '{keyword_principal}'")
        print("\n💡 Puedes continuar con el pipeline usando esta keyword.")
    else:
        print("❌ No se pudo generar una keyword. Verifica la configuración de Gemini.")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

⏭️  Método no seleccionado. Pasa a la siguiente sección.


---

# 🔑 MÉTODO 2: Generación por Keyword

Define directamente la keyword con la que quieres trabajar.

In [ ]:
if metodo_seleccionado == "keyword":
    # Define tu keyword técnica aquí
    keyword_principal = "mlops best practices for llm deployment"
    
    print(f"🔑 Keyword definida: '{keyword_principal}'")
    print("\n💡 Continuando con el pipeline de investigación...")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

⏭️  Método no seleccionado. Pasa a la siguiente sección.


---

# 📰 MÉTODO 3: Generación desde Newsletter

Tienes 2 opciones:
- **Opción A**: Pegar URL pública de newsletter
- **Opción B**: Usar Gmail API para leer newsletters de tu bandeja de entrada (🆕)

## ⚠️ IMPORTANTE: Formatos de URL de Gmail

Gmail usa diferentes tipos de IDs dependiendo de cómo accedas al email:

✅ **FORMATOS COMPATIBLES:**
- `https://mail.google.com/mail/u/0/?permmsgid=msg-f:1849085179733622850` (clic derecho → Copiar enlace)
- Usar la celda "Listar Newsletters" (más abajo) para obtener IDs correctos

❌ **FORMATO NO COMPATIBLE:**
- `https://mail.google.com/mail/u/0/#inbox/FMfcgzQcqthzZXTKZTwjsKffFKdKCpsx` (acceso directo desde inbox)

**Solución recomendada:** Ejecuta primero la celda "Listar Newsletters Disponibles" para ver todos tus newsletters con IDs válidos.

---

### 📋 Listar Newsletters Disponibles en Gmail (RECOMENDADO)

**Ejecuta PRIMERO esta celda** para obtener los IDs correctos de tus newsletters.

Esta celda te mostrará:
- ✅ Subject de cada newsletter
- ✅ Remitente
- ✅ Fecha
- ✅ **ID correcto de Gmail API** (compatible con el sistema)
- ✅ URL lista para copiar

💡 Copia el ID que te interese y pégalo en la variable `gmail_url` de la celda anterior.

In [6]:
# ========================================
# 📋 LISTAR NEWSLETTERS DISPONIBLES
# ========================================
# Ejecuta PRIMERO esta celda para obtener IDs correctos de Gmail API

print("=" * 70)
print("📧 LISTANDO NEWSLETTERS DESDE GMAIL API")
print("=" * 70)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🔧 OPCIONES DE FILTRADO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# OPCIÓN 1: Buscar solo en bandeja "Principal" (excluye Promociones, Social)
only_primary = True  # True = solo Principal | False = todas las bandejas

# OPCIÓN 2: Filtrar por remitentes específicos (más preciso)
# Puedes usar un remitente único o una lista de remitentes
sender_filter = None  # Opciones:
# sender_filter = None                                    # Todos los remitentes
# sender_filter = "newsletter@substack.com"               # Remitente único
# sender_filter = ["newsletter@substack.com", "hello@example.com"]  # Lista

# 💡 REMITENTES COMUNES DE NEWSLETTERS TECH (descomenta para usar):
# sender_filter = [
#     "news@morning.com",           # Morning Brew
#     "hello@substack.com",          # Substack newsletters
#     "newsletter@deeplearning.ai",  # DeepLearning.AI
#     "hello@tldr.tech",             # TLDR Newsletter
#     "team@hackernewsletter.com",   # Hacker Newsletter
#     "newsletter@github.com",       # GitHub Digest
#     "updates@linkedin.com",        # LinkedIn
# ]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

max_results = 20  # Cantidad de newsletters a mostrar

print("\n💡 Configuración de filtrado:")
print(f"   • Bandeja: {'PRINCIPAL (excluye Promociones)' if only_primary else 'TODAS'}")
if sender_filter:
    if isinstance(sender_filter, list):
        print(f"   • Remitentes: {len(sender_filter)} configurados")
        for s in sender_filter[:3]:  # Mostrar primeros 3
            print(f"      - {s}")
        if len(sender_filter) > 3:
            print(f"      ... y {len(sender_filter) - 3} más")
    else:
        print(f"   • Remitente: {sender_filter}")
else:
    print(f"   • Remitentes: TODOS")
print(f"   • Límite: {max_results} newsletters\n")

newsletters = core.list_gmail_newsletters(
    sender_filter=sender_filter,
    max_results=max_results,
    only_primary=only_primary
)

if newsletters:
    print("\n" + "="*70)
    print("📬 NEWSLETTERS DISPONIBLES")
    print("="*70)
    
    for i, nl in enumerate(newsletters, 1):
        print(f"\n{i}. 📨 {nl['subject'][:70]}")  # Aumentado a 70 chars
        print(f"   📤 De: {nl['from'][:60]}")
        print(f"   📅 Fecha: {nl['date']}")
        print(f"   🔑 ID de API: {nl['id']}")
    
    print("\n" + "="*70)
    print("\n✅ CÓMO USAR:")
    print("   1. Copia el 'ID de API' del newsletter que quieras")
    print("   2. Pégalo en la variable 'gmail_url' de la celda 'MÉTODO 3: Newsletter'")
    print("   3. Ejecuta esa celda nuevamente")
    print("\n💡 Ejemplo: gmail_url = '{}'".format(newsletters[0]['id'] if newsletters else 'ID_AQUI'))
    
    # Resumen de remitentes únicos encontrados
    unique_senders = set(nl['from'] for nl in newsletters)
    print(f"\n📊 Resumen: {len(newsletters)} newsletters de {len(unique_senders)} remitentes distintos")
    
else:
    print("⚠️  No se encontraron newsletters con estos filtros")
    print("\n💡 Prueba:")
    print("   • Cambiar only_primary = False (incluir todas las bandejas)")
    print("   • Eliminar sender_filter (mostrar todos los remitentes)")
    print("   • Aumentar max_results (mostrar más resultados)")

📧 LISTANDO NEWSLETTERS DESDE GMAIL API

💡 Configuración de filtrado:
   • Bandeja: PRINCIPAL (excluye Promociones)
   • Remitentes: TODOS
   • Límite: 20 newsletters

🔑 Usando archivo de credenciales: client_secret_385514742115-gq9sltrud83gdn2fsq4n7kiihbgqur8e.apps.googleusercontent.com.json
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=385514742115-dlah24plcrhqb088rfior3h1immv3obi.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A50115%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=UActY625lJ3O0i5lSsUXYUCgiHrAwe&access_type=offline
✅ Autenticación exitosa con Gmail API
🔍 Buscando en BANDEJA PRINCIPAL (excluye Promociones/Social)
💡 Query completa: category:primary

✅ Autenticación exitosa con Gmail API
🔍 Buscando en BANDEJA PRINCIPAL (excluye Promociones/Social)
💡 Query completa: category:primary

📧 Encontrados 20 newsletters
📧 Encontrados 20 newsletters

📬 NEWSLETTERS 

In [8]:
if metodo_seleccionado == "newsletter":
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # 🔧 ELIGE TU MÉTODO DE ACCESO AL NEWSLETTER
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    usar_gmail = True  # True = Gmail API | False = URL pública
    
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    newsletter_analysis = None
    
    if usar_gmail:
        print("📧 MÉTODO: Gmail API")
        print("="*70)
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # 📝 PEGA AQUÍ EL ID DEL NEWSLETTER
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # IMPORTANTE: Usa el ID que obtienes de la celda "Listar Newsletters"
        # Debe ser un ID hexadecimal como: 193d8a1e2b3c4d5f
        # NO uses URLs con formato: #inbox/FMfcgzQcqthzZXTKZTwjsKffFKdKCpsx
        
        gmail_url = "19aba085f615d435"  # ← EDITA AQUÍ
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        if gmail_url == "PEGA_AQUI_EL_ID_DE_LA_CELDA_ANTERIOR":
            print("⚠️  CONFIGURACIÓN REQUERIDA:")
            print("="*70)
            print("\n1️⃣  Ejecuta la celda 'Listar Newsletters' (más abajo)")
            print("2️⃣  Copia el 'ID de API' del newsletter que quieras")
            print("3️⃣  Pégalo en la variable 'gmail_url' de esta celda")
            print("4️⃣  Vuelve a ejecutar esta celda")
            print("\n💡 El ID debe ser hexadecimal (e.g., 193d8a1e2b3c4d5f)")
        else:
            print(f"📰 ID de Gmail: {gmail_url}")
            print("\n🔍 Extrayendo newsletter desde Gmail...\n")
            
            newsletter_analysis = core.extract_newsletter_from_gmail(gmail_url, ghen_context)
    
    else:
        print("🌐 MÉTODO: URL Pública")
        print("="*70)
        
        # Opción 2: URL pública de newsletter
        newsletter_url = "https://mail.google.com/mail/u/0/?ui=2&ik=9f373130bb&view=lg&permmsgid=msg-f:1849748569185440821"
        
        print(f"📰 URL de newsletter: {newsletter_url}")
        print("\n🔍 Analizando el contenido técnico...\n")
        
        newsletter_analysis = core.extract_and_summarize_url(newsletter_url, ghen_context)
    
    # Mostrar resultados
    if newsletter_analysis:
        print("\n" + "="*70)
        print("📊 ANÁLISIS DE LA NEWSLETTER TÉCNICA")
        print("="*70)
        print(newsletter_analysis['raw_analysis'])
        print("="*70)
        
        # Guardar el análisis
        with open('outputs/analisis_newsletter.md', 'w', encoding='utf-8') as f:
            f.write(newsletter_analysis['raw_analysis'])
        
        print("\n💾 Análisis guardado en 'outputs/analisis_newsletter.md'")
        print("\n💡 Usa los 'Temas Sugeridos' para generar contenido técnico.")
    else:
        print("❌ No se pudo analizar el contenido.")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

📧 MÉTODO: Gmail API
📰 ID de Gmail: 19aba085f615d435

🔍 Extrayendo newsletter desde Gmail...

📧 Extrayendo newsletter con ID: 19aba085f615d435
✅ Autenticación exitosa con Gmail API
✅ Newsletter obtenido: 'The Underrated Science of LLM Samplers...'
   De: Decoding AI Magazine <decodingml@substack.com>
   Contenido: 22001 caracteres (texto), 156998 caracteres (HTML)
✅ Newsletter obtenido: 156998 caracteres
📋 Asunto: The Underrated Science of LLM Samplers
📨 De: Decoding AI Magazine <decodingml@substack.com>
✅ Newsletter obtenido: 'The Underrated Science of LLM Samplers...'
   De: Decoding AI Magazine <decodingml@substack.com>
   Contenido: 22001 caracteres (texto), 156998 caracteres (HTML)
✅ Newsletter obtenido: 156998 caracteres
📋 Asunto: The Underrated Science of LLM Samplers
📨 De: Decoding AI Magazine <decodingml@substack.com>
✅ Análisis del newsletter completado

📊 ANÁLISIS DE LA NEWSLETTER TÉCNICA
**Disclaimer:** El contenido del newsletter técnico proporcionado en el prompt estaba in

---

# 🔬 PASO 1: Investigación de Keywords (Opcional)

Si elegiste método 1 o 2, puedes hacer scraping de keywords relacionadas.

In [5]:
# Solo ejecutar si NO estás usando el método newsletter
if metodo_seleccionado in ["topico", "keyword"]:
    print(f"🔍 Scrapeando keywords relacionadas con: '{keyword_principal}'")
    
    # Configuración del scraper
    scraper_config = {
        'keyword': keyword_principal,
        'language': 'es',
        'country': 'es',
        'scrape_levels': 1,  # Nivel de profundidad
        'headless': True
    }
    
    # Crear instancia del scraper
    kw_scraper = scraper.GoogleKeywordScraper(**scraper_config)
    
    # Ejecutar scraping (método correcto: scrape(), no run())
    keywords_df = kw_scraper.scrape()
    
    if keywords_df is not None and not keywords_df.empty:
        # Guardar resultados
        keywords_df.to_csv('outputs/keywords_scraped.csv', index=False)
        print(f"\n✅ {len(keywords_df)} keywords encontradas y guardadas")
        print("\n📋 Primeras 10 keywords:")
        print(keywords_df.head(10))
    else:
        print("⚠️  No se encontraron keywords. Continuando sin ellas.")
else:
    print("⏭️  Saltando investigación de keywords (método newsletter).")

⏭️  Saltando investigación de keywords (método newsletter).


---

# 🌐 PASO 2: Búsqueda de Artículos de Referencia

Busca artículos relacionados para usar como contexto.

In [5]:
if metodo_seleccionado in ["topico", "keyword"]:
    print(f"🔎 Buscando artículos sobre: '{keyword_principal}'")
    
    # Buscar en Google Custom Search
    search_results = core.google_custom_search(
        query=keyword_principal,
        country='ES',
        max_results=5
    )
    
    if not search_results.empty:
        search_results.to_csv('outputs/search_results.csv', index=False)
        print(f"\n✅ {len(search_results)} artículos encontrados")
        print("\n📰 Artículos encontrados:")
        for idx, row in search_results.iterrows():
            print(f"   {idx+1}. {row['title']}")
            print(f"      {row['link']}\n")
    else:
        print("⚠️  No se encontraron artículos.")
else:
    print("⏭️  Saltando búsqueda de artículos (método newsletter).")

⏭️  Saltando búsqueda de artículos (método newsletter).


---

# 📚 PASO 3: Scraping de Contenido de Artículos

In [6]:
if metodo_seleccionado in ["topico", "keyword"]:
    if not search_results.empty:
        print("📖 Scrapeando contenido de los artículos...\n")
        
        scraped_content = []
        
        for idx, row in search_results.iterrows():
            url = row['link']
            print(f"   Scrapeando {idx+1}/{len(search_results)}: {row['title'][:60]}...")
            
            content = core.scrape_article(url)
            
            if content and len(content) > 100:
                scraped_content.append({
                    'title': row['title'],
                    'url': url,
                    'content': content[:3000]  # Limitar a 3000 caracteres
                })
        
        # Guardar contenido scrapeado
        scraped_df = pd.DataFrame(scraped_content)
        if not scraped_df.empty:
            scraped_df.to_csv('outputs/scraped_articles.csv', index=False)
            print(f"\n✅ {len(scraped_df)} artículos scrapeados exitosamente")
        else:
            print("⚠️  No se pudo scrapear ningún artículo")
    else:
        print("⚠️  No hay artículos para scrapear")
        scraped_content = []
else:
    print("⏭️  Usando contenido de newsletter como contexto.")
    scraped_content = []

⏭️  Usando contenido de newsletter como contexto.


---

# ✍️ PASO 4: Generación del Artículo Técnico Final

Genera un artículo técnico completo usando toda la información recopilada.

In [9]:
print("🎨 Preparando contexto para generación del artículo técnico...\n")

# Preparar contexto según el método
context_sources = []

if metodo_seleccionado == "newsletter":
    # Usar el análisis del newsletter como contexto
    if newsletter_analysis:
        # IMPORTANTE: Usar TANTO el análisis COMO el contenido original
        context_sources.append(f"# ANÁLISIS DE LA NEWSLETTER TÉCNICA\n\n{newsletter_analysis['raw_analysis']}")
        context_sources.append(f"# CONTENIDO COMPLETO DE LA NEWSLETTER\n\n{newsletter_analysis['original_content']}")
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # 🔧 CONFIGURA AQUÍ LA KEYWORD PARA EL ARTÍCULO
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # Opciones:
        # 1. Dejar vacío ("") para usar keyword por defecto
        # 2. Usar uno de los "Temas Sugeridos" del análisis anterior
        # 3. Definir tu propia keyword personalizada
        
        keyword_personalizada = ""  # ← EDITA AQUÍ
        
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        
        if not keyword_personalizada:
            keyword_principal = "News: Actualidad y tendencias en IA"
            print("💡 Usando keyword por defecto para artículo de Actualidad:")
            print(f"   '{keyword_principal}'")
        else:
            keyword_principal = keyword_personalizada
            print(f"✅ Usando keyword personalizada: '{keyword_principal}'")
        
        print(f"\n📋 Contexto preparado:")
        print(f"   • Análisis del newsletter: {len(newsletter_analysis['raw_analysis'])} caracteres")
        print(f"   • Contenido original: {len(newsletter_analysis['original_content'])} caracteres")
        print(f"   • Total de contexto: {sum(len(s) for s in context_sources)} caracteres")
    else:
        print("❌ No hay análisis de newsletter disponible")
        keyword_principal = None
else:
    # Usar artículos scrapeados como contexto (métodos "topico" o "keyword")
    if scraped_content:
        for article in scraped_content:
            context_sources.append(f"# {article['title']}\n\n{article['content']}")
        print(f"✅ Usando {len(scraped_content)} artículos como contexto")
    else:
        print("⚠️  No hay artículos scrapeados. Generando solo con la keyword.")

# Generar artículo técnico
if keyword_principal and context_sources:
    print(f"\n🚀 Generando artículo técnico sobre: '{keyword_principal}'\n")
    
    articulo = core.generate_article_with_context(
        keyword=keyword_principal,
        context_sources=context_sources,
        leo_context=ghen_context
    )
    
    if articulo:
        # 🆕 LIMPIAR META-TEXTO del inicio
        print("\n🧹 Limpiando meta-texto del artículo...")
        articulo = core.clean_article_metatext(articulo)
        
        # 🆕 AGREGAR REFERENCIAS AUTOMÁTICAMENTE (filtra tracking URLs)
        print("\n🔗 Agregando referencias a las fuentes...")
        articulo = core.add_source_links_to_article(articulo, context_sources)
        
        # Guardar artículo
        with open('outputs/articulo_ghen_generado.md', 'w', encoding='utf-8') as f:
            f.write(articulo)
        
        print("\n" + "="*70)
        print("📝 ARTÍCULO TÉCNICO GENERADO")
        print("="*70)
        print(articulo[:1000] + "\n...\n")
        print("="*70)
        print(f"\n💾 Artículo completo guardado en 'outputs/articulo_ghen_generado.md'")
        print(f"📊 Longitud: {len(articulo)} caracteres, ~{len(articulo.split())} palabras")
    else:
        print("❌ Error al generar el artículo")
else:
    print("❌ Falta keyword o contexto para generar el artículo")

🎨 Preparando contexto para generación del artículo técnico...

💡 Usando keyword por defecto para artículo de Actualidad:
   'News: Actualidad y tendencias en IA'

📋 Contexto preparado:
   • Análisis del newsletter: 5254 caracteres
   • Contenido original: 156998 caracteres
   • Total de contexto: 162328 caracteres

🚀 Generando artículo técnico sobre: 'News: Actualidad y tendencias en IA'

🎨 Generando artículo de actualidad con contexto LEO...
✅ Artículo generado (15865 caracteres)

🧹 Limpiando meta-texto del artículo...
✅ Meta-texto eliminado del artículo

🔗 Agregando referencias a las fuentes...
✅ Agregadas 86 referencias al artículo

📝 ARTÍCULO TÉCNICO GENERADO
# La ciencia subestimada de los samplers en LLMs: optimización para entornos de producción

El ecosistema de la inteligencia artificial generativa evoluciona a un ritmo vertiginoso, con nuevos modelos y arquitecturas emergiendo constantemente. Sin embargo, en la carrera por desarrollar LLMs más grandes y potentes, un component

---

# 📊 Resumen Final

In [10]:
print("="*70)
print("📊 RESUMEN DEL PROCESO")
print("="*70)
print(f"\n🎯 Método utilizado: {metodo_seleccionado.upper()}")
print(f"🔑 Keyword principal: {keyword_principal}")

if metodo_seleccionado in ["topico", "keyword"]:
    print(f"📚 Artículos scrapeados: {len(scraped_content) if scraped_content else 0}")
elif metodo_seleccionado == "newsletter":
    print(f"📰 Newsletter analizada: {'Sí' if newsletter_analysis else 'No'}")

print(f"\n✅ Archivos generados en outputs/:")
for file in ['articulo_ghen_generado.md', 'analisis_newsletter.md', 'keywords_scraped.csv', 
             'search_results.csv', 'scraped_articles.csv']:
    filepath = f'outputs/{file}'
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        print(f"   • {file} ({size:,} bytes)")

print("\n" + "="*70)
print("🎉 ¡Proceso completado!")
print("="*70)

📊 RESUMEN DEL PROCESO

🎯 Método utilizado: NEWSLETTER
🔑 Keyword principal: News: Actualidad y tendencias en IA
📰 Newsletter analizada: Sí

✅ Archivos generados en outputs/:
   • articulo_ghen_generado.md (42,323 bytes)
   • analisis_newsletter.md (5,345 bytes)
   • keywords_scraped.csv (259 bytes)
   • search_results.csv (1,270 bytes)
   • scraped_articles.csv (23,378 bytes)

🎉 ¡Proceso completado!


---

# 🔍 PASO 4.5: Quality Assurance (Sistema en 2 Fases)

Sistema de evaluación y mejora de calidad del artículo generado.

## 📋 FASE 1: Evaluación Ciega (Objetiva)

**El QA actúa como juez ciego:**
- ✅ Evalúa SOLO el resultado final (sin acceso a fuentes)
- ✅ No tiene sesgos por conocer las instrucciones originales  
- ✅ Mide: narrativa, fluidez, tono, repeticiones, valor, estructura
- ✅ Genera reporte objetivo con problemas específicos detectados

## 🔧 FASE 2: Mejoras Quirúrgicas CON Contexto

**El editor tiene acceso completo al contexto:**
- ✅ Lee el reporte QA con problemas específicos
- ✅ Tiene acceso a fuentes originales (newsletter/artículos)
- ✅ Conoce la personalidad GHEN (tono, experiencias)
- ✅ Aplica SOLO las mejoras mencionadas en el reporte QA
- ✅ Preserva 95%+ del contenido original palabra por palabra

**Validaciones de seguridad (6 capas):**
1. ❌ Detecta si regeneró contenido tipo newsletter/resumen
2. ❌ Rechaza si longitud cambia >30% (debe ser ±5%)
3. ❌ Rechaza si se pierden secciones (>30%)
4. ❌ Rechaza si longitud <2000 chars
5. ❌ Rechaza si no preserva inicio del artículo
6. ❌ Rechaza si similitud >95% (no aplicó mejoras) o <50% (regeneró todo)

**Configuración:**
- `ENABLE_QA`: Activar/desactivar sistema completo
- `APPLY_QA_IMPROVEMENTS`: Auto-mejora con contexto (Fase 2)

---

In [11]:
# ========================================
# CONFIGURACIÓN: Quality Assurance
# ========================================

ENABLE_QA = True  # Cambiar a False para omitir QA
APPLY_QA_IMPROVEMENTS = True  # Cambiar a False para solo ver reporte sin mejorar

# ========================================
# Ejecutar Quality Assurance
# ========================================

if ENABLE_QA and 'articulo' in dir() and articulo:
    print("=" * 70)
    print("🔍 QUALITY ASSURANCE: Evaluación Ciega de Calidad")
    print("=" * 70)
    print("\n💡 FASE 1: Evaluación objetiva (sin acceso a fuentes)")
    print("   Criterios: narrativa, listas vs párrafos, tono, repeticiones, valor, estructura\n")
    
    # Ejecutar QA ciego (solo necesita el artículo)
    qa_report = core.qa_article_coverage(
        article_content=articulo,
        related_keywords_context=None,  # QA ciego no necesita contexto
        main_query=None
    )
    
    if qa_report:
        print("\n" + "=" * 70)
        print("📊 REPORTE QA GENERADO")
        print("=" * 70)
        
        # Mostrar primeras líneas del reporte
        lines = qa_report.split('\n')
        preview_lines = lines[:50] if len(lines) > 50 else lines
        print('\n'.join(preview_lines))
        
        if len(lines) > 50:
            print("\n... (ver reporte completo en outputs/qa_report.md) ...\n")
        
        print("=" * 70)
        print(f"\n💾 Reporte completo guardado en: outputs/qa_report.md")
        
        # Aplicar mejoras automáticas si está habilitado
        if APPLY_QA_IMPROVEMENTS:
            print("\n" + "=" * 70)
            print("🔧 FASE 2: Aplicando Mejoras Quirúrgicas CON Contexto")
            print("=" * 70)
            print("\n💡 FASE 2 tiene acceso al contexto original para:")
            print("   • Mantener precisión técnica de las fuentes")
            print("   • Preservar tono y personalidad GHEN")
            print("   • Aplicar SOLO cambios específicos del QA")
            print("   • Validar que se preserva 95%+ del contenido\n")
            
            # CRÍTICO: Pasar context_sources y ghen_context para mejoras precisas
            articulo_mejorado = core.improve_article_based_on_qa(
                article_content=articulo,
                qa_report=qa_report,
                context_sources=context_sources if 'context_sources' in dir() else None,
                ghen_context=ghen_context if 'ghen_context' in dir() else None
            )
            
            if articulo_mejorado and articulo_mejorado != articulo:
                print("\n📝 GUARDANDO ARTÍCULO MEJORADO...")
                print("=" * 70)
                
                # Guardar versión mejorada con nombre específico
                mejora_file = 'outputs/articulo_ghen_mejorado_qa.md'
                with open(mejora_file, 'w', encoding='utf-8') as f:
                    f.write(articulo_mejorado)
                
                print(f"✅ Guardado en: {mejora_file}")
                print(f"   📏 Longitud: {len(articulo_mejorado):,} caracteres (~{len(articulo_mejorado.split())} palabras)")
                
                # TAMBIÉN actualizar el archivo principal para WordPress
                original_file = 'outputs/articulo_ghen_generado.md'
                with open(original_file, 'w', encoding='utf-8') as f:
                    f.write(articulo_mejorado)
                
                print(f"✅ Actualizado también: {original_file}")
                print(f"   (Este es el archivo que WordPress publicará)")
                
                # Actualizar variable en memoria
                articulo = articulo_mejorado
                
                print("\n📊 COMPARACIÓN:")
                print(f"   Original:  {len(articulo)} caracteres")
                print(f"   Mejorado:  {len(articulo_mejorado)} caracteres")
                print(f"   Cambio:    {((len(articulo_mejorado) / len(articulo)) - 1) * 100:+.1f}%")
                
                print("\n✅ PROCESO COMPLETO (2 FASES):")
                print("   1. ✅ QA ciego evaluó calidad objetivamente")
                print("   2. ✅ Mejoras quirúrgicas aplicadas CON contexto")
                print("   3. ✅ Validaciones de preservación de contenido (6 capas)")
                print("   4. ✅ Artículo mejorado guardado y actualizado")
                
            else:
                print("\nℹ️  No se aplicaron mejoras (artículo ya cumple estándares de calidad)")
                print("   O las validaciones QA rechazaron los cambios propuestos")
                print("\n💡 Archivo original preservado:")
                print("   • outputs/articulo_ghen_generado.md (sin cambios)")
        else:
            print("\nℹ️  Mejoras automáticas desactivadas (APPLY_QA_IMPROVEMENTS = False)")
            print("   Revisa el reporte QA manualmente para decidir cambios")
    else:
        print("\n❌ Error: No se pudo generar el reporte QA")
else:
    if not ENABLE_QA:
        print("⏭️  Quality Assurance omitido (ENABLE_QA = False)")
    elif 'articulo' not in dir():
        print("⚠️  No hay artículo generado para evaluar")
        print("   Ejecuta primero la celda de generación de contenido (PASO 4)")

🔍 QUALITY ASSURANCE: Evaluación Ciega de Calidad

💡 FASE 1: Evaluación objetiva (sin acceso a fuentes)
   Criterios: narrativa, listas vs párrafos, tono, repeticiones, valor, estructura

🔄 Ejecutando QA del artículo...
✅ Reporte QA generado y guardado en outputs/qa_report.md

📊 REPORTE QA GENERADO
# REPORTE QA

## 1. NARRATIVA Y FLUIDEZ (Puntuación: 8/10)

**Análisis:**
- El texto fluye de manera generalmente natural de un párrafo a otro, siguiendo una progresión lógica desde la introducción del problema hasta las soluciones técnicas y sus implicaciones prácticas.
- Las transiciones son en su mayoría suaves, facilitadas por el uso claro de títulos y subtítulos. La introducción de cada sección suele enlazar bien con el tema general.
- Se siente como una conversación experta, muy informativa y didáctica, aunque ocasionalmente se acerca más a una exposición estructurada que a un diálogo fluido.
- El ritmo en las oraciones es consistente y profesional, pero podría beneficiarse de una mayor

---

# 🎨 PASO 5 (Opcional): Generación de Imagen Destacada

Genera automáticamente una imagen destacada para el artículo usando **Vertex AI Imagen 3**.

**Requisitos previos:**
- Variables de entorno configuradas: `PROJECT_ID`, `LOCATION`, `SERVICE_ACCOUNT_KEY`
- Service account JSON en la raíz del proyecto

**Configuración:**
- `GENERATE_IMAGE`: `True` para generar imagen, `False` para omitir
- Costo: ~$0.04 USD por imagen
- La imagen se guarda en `outputs/featured_image.png`

In [12]:
# ========================================
# CONFIGURACIÓN: Generación de Imagen
# ========================================

GENERATE_IMAGE = True  # Cambiar a False para omitir generación de imagen

# ========================================
# Generación de imagen destacada
# ========================================

featured_image_path = None

if GENERATE_IMAGE:
    try:
        print("=" * 70)
        print("🎨 GENERANDO IMAGEN DESTACADA")
        print("=" * 70)
        print("\n💡 Proceso en 2 fases:")
        print("   1️⃣ Análisis del artículo completo → Extracción de conceptos clave")
        print("   2️⃣ Generación de prompt visual específico → Imagen única")
        print()
        
        # 1. Leer artículo generado (preferir versión mejorada si existe)
        article_files = [
            'outputs/articulo_ghen_mejorado_qa.md',
            'outputs/articulo_ghen_generado.md'
        ]
        
        article_text = None
        article_file_used = None
        
        for f in article_files:
            if os.path.exists(f):
                with open(f, 'r', encoding='utf-8') as file:
                    article_text = file.read()
                    article_file_used = f
                    break
        
        if not article_text:
            print("❌ No se encontró ningún artículo para generar la imagen")
            raise Exception("Artículo no encontrado")
        
        print(f"📄 Usando artículo: {article_file_used}")
        print(f"   Longitud: {len(article_text):,} caracteres\n")
        
        # 2. Generar prompt optimizado con Gemini (análisis completo del artículo)
        from longcontent_generator import generate_featured_image_prompt
        
        image_prompt = generate_featured_image_prompt(
            article_text=article_text,
            article_title=keyword_principal
        )
        
        if image_prompt:
            print("\n" + "=" * 70)
            print("🎨 PROMPT VISUAL GENERADO")
            print("=" * 70)
            print(image_prompt)
            print("=" * 70)
            
            # 3. Generar imagen con Vertex AI Imagen 3
            from longcontent_generator import generate_image_from_prompt, optimize_image_for_wordpress
            
            print("\n📸 Generando imagen con Vertex AI Imagen 3...")
            featured_image_path = generate_image_from_prompt(
                prompt=image_prompt,
                output_path="outputs/featured_image.png"
            )
            
            if featured_image_path:
                # 4. Optimizar para WordPress
                featured_image_path = optimize_image_for_wordpress(featured_image_path)
                
                # Mostrar info del archivo generado
                image_size = os.path.getsize(featured_image_path)
                print(f"\n✅ Imagen destacada generada:")
                print(f"   📁 Archivo: {featured_image_path}")
                print(f"   📏 Tamaño: {image_size:,} bytes (~{image_size//1024}KB)")
                print(f"\n💡 La imagen refleja conceptos específicos del artículo:")
                print(f"   {image_prompt[:200]}...")
            else:
                print("\n⚠️  No se pudo generar la imagen")
        else:
            print("\n⚠️  No se pudo generar el prompt de imagen")
    
    except Exception as e:
        print(f"\n❌ Error al generar imagen: {e}")
        print("   Continuando sin imagen destacada...")
        featured_image_path = None
else:
    print("⏭️  Generación de imagen omitida (GENERATE_IMAGE = False)")

🎨 GENERANDO IMAGEN DESTACADA

💡 Proceso en 2 fases:
   1️⃣ Análisis del artículo completo → Extracción de conceptos clave
   2️⃣ Generación de prompt visual específico → Imagen única

📄 Usando artículo: outputs/articulo_ghen_mejorado_qa.md
   Longitud: 17,566 caracteres

🎨 Generando prompt para imagen destacada...
   🎭 Estilo seleccionado: bauhaus design style
   🎨 Paleta: forest green, burnt sienna, and beige
   📊 Analizando artículo para crear METÁFORA visual única...

   🔍 Extrayendo metáfora creativa...
   ✅ Metáfora extraída:
1.  Un vasto y complejo taller de relojería, donde cada engranaje, resorte y piñón está intrincadamente forjado con líneas de código lumínicas. El aire resuena con la precisión de un sistema perfectamente sincronizado, que representa una base de software masiva y en constante evolución.
2.  Una figura etérea y diminuta, un maestro relojero de luz pura, que se cierne sobre la maquinaria. Sin tocarla, su presencia irradia una inteligencia que alinea instantánea

---

# 📰 PASO 6 (Opcional): Publicación en WordPress

Publica el artículo generado directamente en WordPress como borrador o publicado.

In [13]:
# ========================================
# CONFIGURACIÓN: Publicación en WordPress
# ========================================

# IMPORTANTE: Actualiza estos valores antes de publicar
ARTICLE_TITLE = "LAB: Actualidad IA"  # Cambiar por el título real
PUBLICATION_STATUS = "draft"  # Opciones: 'draft' o 'publish'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🔍 DETECCIÓN INTELIGENTE DEL ARTÍCULO A PUBLICAR
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os
from datetime import datetime

# Prioridad de archivos (del más reciente al más antiguo)
possible_files = [
    "outputs/articulo_ghen_mejorado_qa.md",  # Con mejoras QA aplicadas
    "outputs/articulo_mejorado_qa.md",       # Nombre alternativo
    "outputs/articulo_ghen_generado.md",     # Original generado
    "outputs/articulo_completo.md"           # Fallback legacy
]

# Buscar el archivo más reciente que exista
MARKDOWN_FILE = None
file_info = []

for file_path in possible_files:
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        file_mtime = os.path.getmtime(file_path)
        file_date = datetime.fromtimestamp(file_mtime)
        
        file_info.append({
            'path': file_path,
            'size': file_size,
            'modified': file_date
        })

# Seleccionar el más reciente con contenido válido (> 1000 chars)
valid_files = [f for f in file_info if f['size'] > 1000]

if valid_files:
    # Ordenar por fecha de modificación (más reciente primero)
    valid_files.sort(key=lambda x: x['modified'], reverse=True)
    MARKDOWN_FILE = valid_files[0]['path']
    
    print("🔍 DETECCIÓN DE ARCHIVO A PUBLICAR")
    print("=" * 70)
    print(f"\n✅ Archivo seleccionado: {MARKDOWN_FILE}")
    print(f"   📅 Última modificación: {valid_files[0]['modified'].strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   📏 Tamaño: {valid_files[0]['size']:,} bytes (~{valid_files[0]['size']//1000}K)")
    
    # Mostrar otros archivos disponibles
    if len(valid_files) > 1:
        print(f"\n📋 Otros archivos disponibles:")
        for f in valid_files[1:]:
            age_minutes = (valid_files[0]['modified'] - f['modified']).total_seconds() / 60
            print(f"   • {f['path']}")
            print(f"     Modificado hace {age_minutes:.0f} min, tamaño: {f['size']:,} bytes")
    
    # Advertencia si el archivo es muy antiguo
    age_minutes = (datetime.now() - valid_files[0]['modified']).total_seconds() / 60
    if age_minutes > 30:
        print(f"\n⚠️  ADVERTENCIA: El archivo tiene {age_minutes:.0f} minutos de antigüedad")
        print("   ¿Has ejecutado las celdas de generación/QA recientemente?")
else:
    print("❌ ERROR: No se encontró ningún artículo válido para publicar")
    print("\n📋 Archivos encontrados:")
    for f in file_info:
        print(f"   • {f['path']} - {f['size']} bytes (demasiado corto)")
    print("\n💡 Ejecuta primero las celdas de generación de contenido (PASO 4 y 4.5)")

# NOTA: Las credenciales se cargan automáticamente desde .env
# Necesitas tener configuradas las variables:
#   WORDPRESS_LOGIN_GHEN
#   WORDPRESS_PASSWORD_GHEN

if MARKDOWN_FILE:
    print(f"\n📝 Título configurado: {ARTICLE_TITLE}")
    print(f"📊 Estado: {PUBLICATION_STATUS}")
    print(f"🖼️  Imagen destacada: {'Sí' if 'featured_image_path' in dir() and featured_image_path else 'No'}")

    # ========================================
    # Publicar en WordPress
    # ========================================

    from longcontent_generator import publish_article_from_markdown_cleaned

    print("\n" + "=" * 70)
    print("📤 PUBLICANDO EN WORDPRESS")
    print("=" * 70)

    # Verificar si tenemos imagen destacada generada
    image_to_use = None
    if 'featured_image_path' in dir() and featured_image_path:
        image_to_use = featured_image_path
        print(f"🖼️  Usando imagen destacada: {featured_image_path}")

    # Publicar artículo (con o sin imagen)
    result = publish_article_from_markdown_cleaned(
        article_title=ARTICLE_TITLE,
        markdown_file_path=MARKDOWN_FILE,
        status=PUBLICATION_STATUS,
        featured_image_path=image_to_use
    )

    print("\n" + "=" * 70)
    if result and result.get('success'):
        print("✅ PUBLICACIÓN EXITOSA")
        print("=" * 70)
        print(f"📝 Post ID: {result['post_id']}")
        print(f"🔗 URL: {result['post_url']}")
        if result.get('featured_media_id'):
            print(f"🖼️  Imagen destacada asignada (ID: {result['featured_media_id']})")
        print("\n💡 El artículo está en modo '{}'. Ve a WordPress para revisarlo.".format(PUBLICATION_STATUS))
    else:
        print("❌ ERROR EN LA PUBLICACIÓN")
        print("=" * 70)
        print("Verifica:")
        print("  • Credenciales en .env (WORDPRESS_LOGIN_GHEN, WORDPRESS_PASSWORD_GHEN)")
        print("  • Conexión a Internet")
        print("  • URL de WordPress (https://ghendigital.com/wp-json/wp/v2/posts)")
else:
    print("\n⏭️  Publicación cancelada - no hay archivo válido para publicar")

🔍 DETECCIÓN DE ARCHIVO A PUBLICAR

✅ Archivo seleccionado: outputs/articulo_mejorado_qa.md
   📅 Última modificación: 2025-11-25 16:59:55
   📏 Tamaño: 42,323 bytes (~42K)

📋 Otros archivos disponibles:
   • outputs/articulo_ghen_generado.md
     Modificado hace 3 min, tamaño: 42,323 bytes
   • outputs/articulo_ghen_mejorado_qa.md
     Modificado hace 152 min, tamaño: 17,873 bytes

📝 Título configurado: LAB: Actualidad IA
📊 Estado: draft
🖼️  Imagen destacada: Sí

📤 PUBLICANDO EN WORDPRESS
🖼️  Usando imagen destacada: outputs/featured_image.png
✅ Archivo Markdown leído: outputs/articulo_mejorado_qa.md
✅ Markdown limpiado y corregido automáticamente
✅ Markdown convertido a HTML correctamente
🖼️  Procesando imagen destacada...
📤 Subiendo imagen a WordPress: featured_image.png
✅ Imagen subida correctamente
   🆔 Media ID: 1200
   🔗 URL: https://ghendigital.com/wp-content/uploads/2025/11/featured_image-7.jpg
🔄 Publicando en WordPress (status: draft)...
✅ Imagen subida correctamente
   🆔 Media 

---

# 📱 PASO 7 (Opcional): Generación de Posts para Redes Sociales

Genera automáticamente posts optimizados para cada red social a partir del artículo publicado.

**Plataformas soportadas:**
- 🐦 **Twitter/X**: Thread de 5-6 tweets con hooks y hashtags
- 💼 **LinkedIn**: Post profesional para CTOs y Tech Leads
- 🔴 **Reddit**: Post técnico sin marketing (valor para comunidad)
- 🧵 **Threads**: Post casual y conversacional

**Requisitos:**
- Artículo generado en `outputs/articulo_ghen_generado.md`
- URL del artículo en WordPress (para incluir en los posts)

**Outputs:**
- `outputs/social/twitter_*.md` - Thread listo para copiar
- `outputs/social/linkedin_*.md` - Post profesional
- `outputs/social/reddit_*.md` - Post + subreddits sugeridos
- `outputs/social/threads_*.md` - Post casual

In [ ]:
# ========================================
# CONFIGURACIÓN: Generación de Posts Sociales
# ========================================

GENERATE_SOCIAL_POSTS = True  # Cambiar a False para omitir

# Plataformas a generar (True/False para cada una)
SOCIAL_PLATFORMS = {
    "twitter": True,     # Thread de 5-6 tweets
    "linkedin": True,    # Post profesional
    "reddit": True,      # Post técnico (publicar manual)
    "threads": False,    # Post casual
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🔧 URL DEL ARTÍCULO EN WORDPRESS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Esta URL se incluirá en todos los posts generados
# Si acabas de publicar, copia la URL del resultado anterior

ARTICLE_URL = ""  # ← PEGA AQUÍ LA URL DE WORDPRESS

# Si dejaste vacío, intentamos obtenerla del resultado de WordPress
if not ARTICLE_URL and 'result' in dir() and result and result.get('post_url'):
    ARTICLE_URL = result['post_url']
    print(f"✅ URL detectada automáticamente: {ARTICLE_URL}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if GENERATE_SOCIAL_POSTS:
    if not ARTICLE_URL:
        print("⚠️  CONFIGURACIÓN REQUERIDA:")
        print("="*70)
        print("\n1️⃣  Publica primero el artículo en WordPress (PASO 6)")
        print("2️⃣  Copia la URL del artículo publicado")
        print("3️⃣  Pégala en la variable 'ARTICLE_URL' de esta celda")
        print("4️⃣  Vuelve a ejecutar esta celda")
        print("\n💡 Ejemplo: ARTICLE_URL = 'https://ghendigital.com/tu-articulo-slug'")
    else:
        print("="*70)
        print("📱 GENERANDO POSTS PARA REDES SOCIALES")
        print("="*70)
        
        # Detectar archivo de artículo
        article_files = [
            "outputs/articulo_ghen_mejorado_qa.md",
            "outputs/articulo_ghen_generado.md"
        ]
        
        article_path = None
        for f in article_files:
            if os.path.exists(f):
                article_path = f
                break
        
        if article_path:
            print(f"\n📄 Artículo: {article_path}")
            print(f"🔗 URL: {ARTICLE_URL}")
            
            # Obtener plataformas activas
            active_platforms = [p for p, active in SOCIAL_PLATFORMS.items() if active]
            print(f"📊 Plataformas: {', '.join(active_platforms)}")
            
            # Importar y generar
            from longcontent_generator.social import generate_all_social_posts
            
            social_posts = generate_all_social_posts(
                article_path=article_path,
                article_url=ARTICLE_URL,
                platforms=active_platforms
            )
            
            if social_posts:
                print("\n" + "="*70)
                print("✅ POSTS GENERADOS EXITOSAMENTE")
                print("="*70)
                
                # Mostrar preview de cada plataforma
                for platform, data in social_posts.items():
                    if not data:
                        continue
                    
                    print(f"\n{'─'*50}")
                    print(f"📌 {platform.upper()}")
                    print('─'*50)
                    
                    if platform == "twitter":
                        thread = data.get("thread", [])
                        print(f"📝 Thread de {len(thread)} tweets")
                        if thread:
                            print(f"\n[Tweet 1 - Hook]")
                            print(thread[0][:200] + "..." if len(thread[0]) > 200 else thread[0])
                        
                    elif platform == "linkedin":
                        post = data.get("post", "")
                        print(f"📝 Post de {len(post)} caracteres")
                        print(f"\n{post[:300]}..." if len(post) > 300 else post)
                        
                    elif platform == "reddit":
                        print(f"📝 Título: {data.get('title', '')[:80]}")
                        print(f"📋 Subreddits: {', '.join(data.get('suggested_subreddits', []))}")
                        
                    elif platform == "threads":
                        post = data.get("post", "")
                        print(f"📝 Post de {len(post)} caracteres")
                        print(f"\n{post[:200]}..." if len(post) > 200 else post)
                
                print("\n" + "="*70)
                print("💾 ARCHIVOS GUARDADOS EN: outputs/social/")
                print("="*70)
                
                # Listar archivos generados
                social_dir = "outputs/social"
                if os.path.exists(social_dir):
                    for f in os.listdir(social_dir):
                        if f.endswith('.md') and not f.startswith('README'):
                            filepath = os.path.join(social_dir, f)
                            size = os.path.getsize(filepath)
                            print(f"   • {f} ({size:,} bytes)")
                
                print("\n💡 PRÓXIMOS PASOS:")
                print("   1. Revisa cada archivo en outputs/social/")
                print("   2. Copia el contenido a cada red social")
                print("   3. ⚠️ Reddit: publica MANUALMENTE (evita bans)")
            else:
                print("\n❌ Error al generar posts")
        else:
            print("\n❌ No se encontró artículo generado")
            print("   Ejecuta primero las celdas de generación (PASO 4)")
else:
    print("⏭️  Generación de posts sociales omitida (GENERATE_SOCIAL_POSTS = False)")